# Image Classifier (FashionMNIST)

PyTorch + TorchVision image classifier, assembled from Tasks 1-8. Cells share one kernel namespace, so later cells use variables/objects defined by earlier cells directly, with no cross-file imports.

## Task 1 - downloader.py: download and split the dataset

In [ ]:
# Import torch for manual_seed and random_split
import torch
from torch.utils.data import random_split

# Import torchvision datasets module to load FashionMNIST and ToTensor to convert images to tensors
from torchvision import datasets
from torchvision.transforms import ToTensor

# Report status: beginning download of the FashionMNIST training/validation dataset
print("Downloading FashionMNIST training data (train=True)...")

# Download FashionMNIST training split (60000 samples) into train_and_valid_data, converting images to tensors
train_and_valid_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

# Report status: training/validation dataset downloaded, with total sample count
print(f"train_and_valid_data ready. Total samples: {len(train_and_valid_data)}")

# Report status: beginning download of the FashionMNIST test dataset
print("Downloading FashionMNIST test data (train=False)...")

# Download FashionMNIST test split (10000 samples) into test_data, converting images to tensors
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

# Report status: test dataset downloaded, with total sample count
print(f"test_data ready. Total samples: {len(test_data)}")

# Manually set the random seed to 42 so the train/validation split is reproducible
torch.manual_seed(42)

# Report status: seed has been set
print("Random seed manually set to 42.")

# Split train_and_valid_data into 55000 training samples and 5000 validation samples
train_data, valid_data = random_split(train_and_valid_data, [55000, 5000])

# Report status: split complete, with resulting sample counts for each subset
print(f"Split complete. train_data: {len(train_data)} samples, valid_data: {len(valid_data)} samples")

## Task 2 - loader.py: build DataLoaders and inspect a training sample

In [ ]:
# Import torch and DataLoader to batch and iterate over the datasets
import torch
from torch.utils.data import DataLoader

# Manually set the random seed to 42
torch.manual_seed(42)

# Load the training set with batch size 32, shuffling enabled
train_dataloader = DataLoader(train_data, batch_size=32, shuffle=True)

# Load the validation set with batch size 32, no shuffling
valid_dataloader = DataLoader(valid_data, batch_size=32, shuffle=False)

# Load the testing set with batch size 32, no shuffling
test_dataloader = DataLoader(test_data, batch_size=32, shuffle=False)

# Each sample in train_data is a (image, target) tuple; assign its first element's parts to X_sample and y_sample
X_sample, y_sample = train_data[0]

# Print X_sample's shape and data type
print(f"X_sample shape: {X_sample.shape}, dtype: {X_sample.dtype}")

# Look up the class name for y_sample using the class labels from train_and_valid_data
print(f"y_sample: {y_sample}, class: {train_and_valid_data.classes[y_sample]}")

## Task 3 - checker.py: inspect train_data[0]

In [ ]:
# Unpack the first sample of train_data into X_sample (image) and y_sample (target)
X_sample, y_sample = train_data[0]

# Look up the class name for y_sample from the classes list
class_name = train_and_valid_data.classes[y_sample]

# Print the sample's data type, shape, and class_name
print(f"dtype: {X_sample.dtype}, shape: {X_sample.shape}, class_name: {class_name}")

## Task 4 - imageclassifier.py: model architecture and device selection

In [ ]:
# Import torch to check CUDA availability and torch's neural network module to build the model
import torch
from torch import nn

# Select cuda as the device if available, otherwise fall back to cpu
device = "cuda" if torch.cuda.is_available() else "cpu"


# Define the image classifier as a subclass of nn.Module
class ImageClassifier(nn.Module):
    # Define the layers used in the forward pass
    def __init__(self):
        # Initialize the parent nn.Module class
        super().__init__()

        # Flatten each (1, 28, 28) image tensor into a 784-length vector
        self.flatten = nn.Flatten()

        # Define the hidden layer stack: 784 -> 300 -> 100 -> 10, with ReLU between hidden layers
        self.linear_relu_stack = nn.Sequential(
            # First hidden layer: 784 input features to 300 neurons
            nn.Linear(784, 300),
            # ReLU activation after the first hidden layer
            nn.ReLU(),
            # Second hidden layer: 300 neurons to 100 neurons
            nn.Linear(300, 100),
            # ReLU activation after the second hidden layer
            nn.ReLU(),
            # Output layer: 100 neurons to 10 class logits
            nn.Linear(100, 10),
        )

    # Define how input data flows through the layers
    def forward(self, x):
        # Flatten the input image tensor
        x = self.flatten(x)

        # Pass the flattened tensor through the hidden layer stack to get logits
        logits = self.linear_relu_stack(x)

        # Return the raw logits (no activation applied)
        return logits

## Task 5 - training.py: train with SGD and torchmetrics

In [ ]:
# Import torch for the optimizer and torchmetrics for the accuracy metric
import torch
from torch import nn
import torchmetrics

# Instantiate the model and move it to the selected device
model = ImageClassifier().to(device)

# Define the loss function: cross-entropy loss for multi-class classification on raw logits
loss_fn = nn.CrossEntropyLoss()

# Define the optimizer: Stochastic Gradient Descent with learning rate 0.001, no momentum
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

# Define the accuracy metric (10 FashionMNIST classes) and move it to the selected device
accuracy_metric = torchmetrics.classification.MulticlassAccuracy(num_classes=10).to(device)

# Set the number of training epochs
epochs = 20

# Track training accuracy across epochs for later plotting
accuracy_history = []

# Report status: training is starting, on which device
print(f"Starting training on device: {device}")

# Loop over the dataset multiple times (one full pass per epoch)
for epoch in range(epochs):
    # Set the model to training mode (enables dropout/batchnorm training behavior if present)
    model.train()

    # Track the running loss total for this epoch
    running_loss = 0.0

    # Iterate over batches of (image, target) pairs from the training DataLoader
    for X, y in train_dataloader:
        # Move the batch to the selected device
        X, y = X.to(device), y.to(device)

        # Reset gradients from the previous step
        optimizer.zero_grad()

        # Forward pass: compute predicted logits
        pred = model(X)

        # Compute the loss between predictions and targets
        loss = loss_fn(pred, y)

        # Backward pass: compute gradients
        loss.backward()

        # Update model parameters using the computed gradients
        optimizer.step()

        # Accumulate this batch's loss, weighted by batch size
        running_loss += loss.item() * X.size(0)

        # Update the running accuracy metric with this batch's predictions and targets
        accuracy_metric.update(pred, y)

    # Compute the average loss over all training samples this epoch
    epoch_loss = running_loss / len(train_dataloader.dataset)

    # Compute the accuracy accumulated over the whole epoch
    epoch_accuracy = accuracy_metric.compute()

    # Record this epoch's accuracy for later plotting
    accuracy_history.append(epoch_accuracy.item())

    # Report status: epoch number, average loss, and accuracy
    print(f"Epoch {epoch + 1}/{epochs} - loss: {epoch_loss:.4f} - accuracy: {epoch_accuracy:.4f}")

    # Reset the accuracy metric so it doesn't accumulate across epochs
    accuracy_metric.reset()

# Report status: training loop complete
print("Training complete.")

## Task 6 - evaluation.py: evaluate on a small batch

In [ ]:
# Import torch for the no_grad inference context
import torch

# Set the model to evaluation mode (disables dropout/batchnorm training behavior if present)
model.eval()

# Pull one batch of (images, targets) from the validation DataLoader
X_batch, y_batch = next(iter(valid_dataloader))

# Take a small batch of 3 samples from that batch
X_small, y_small = X_batch[:3], y_batch[:3]

# Move the small batch to the selected device
X_small, y_small = X_small.to(device), y_small.to(device)

# Report status: evaluating the small batch, on which device
print(f"Evaluating a small batch of {X_small.size(0)} samples on device: {device}")

# Disable gradient tracking since this is inference, not training
with torch.no_grad():
    # Forward pass: compute predicted logits for the small batch
    logits = model(X_small)

# Report status: evaluation forward pass complete, with the resulting logits shape
print(f"Evaluation complete. logits shape: {logits.shape}")

## Task 7 - probabilities.py: softmax and top-4 probabilities

In [ ]:
# Import torch for softmax/topk operations
import torch

# Convert logits to probabilities using softmax across the class dimension
probabilities = torch.softmax(logits, dim=1)

# Pick out the top 4 probabilities (and their class indices) for each sample
top4_probs, top4_indices = torch.topk(probabilities, k=4, dim=1)

# Print the true label and the top-4 predicted classes with probabilities, for each sample
for i in range(X_small.size(0)):
    # Look up the true class name for this sample
    true_class = train_and_valid_data.classes[y_small[i]]

    # Build a list of (class_name, probability) pairs for this sample's top-4 predictions
    top4 = [
        (train_and_valid_data.classes[idx], prob.item())
        for prob, idx in zip(top4_probs[i], top4_indices[i])
    ]

    # Print the true class alongside the top-4 predictions
    print(f"Sample {i}: true class = {true_class}, top-4 predictions = {top4}")

## Task 8 - plot_accuracy.py: plot training accuracy per epoch

In [ ]:
# Import matplotlib's plotting interface
import matplotlib.pyplot as plt

# Render plots inline in the notebook
%matplotlib inline

# Build the list of epoch numbers (1-indexed) to use as the x-axis
epoch_numbers = list(range(1, epochs + 1))

# Plot training accuracy against epoch number
plt.plot(epoch_numbers, accuracy_history, marker="o")

# Label the x-axis
plt.xlabel("Epoch")

# Label the y-axis
plt.ylabel("Training Accuracy")

# Give the plot a title
plt.title("Training Accuracy per Epoch")

# Display the plot
plt.show()